In [1]:
import psutil
ram = psutil.virtual_memory()
print(f"Available RAM: {ram.available / 1e9:.1f} GB")
# You need at least 3GB free before loading the model

Available RAM: 12.9 GB


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [3]:

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available for this GPU: {torch.cuda.is_available()}")
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print("\nDownloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Loading model on CPU (this takes 1-2 minutes)")
model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="cpu",
        dtype=torch.float32,
        low_cpu_mem_usage=True
    )

model.eval()
print("Model loaded successfully")
    
prompt ="<|system|>You are a helpful assistant.</s><|user|> What is Linux Mint?</s><|assistant|>"
inputs = tokenizer(prompt, return_tensors="pt")

print("\nGenerating response...")
with torch.no_grad():
    outputs = model.generate(
        **inputs, max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n--- Response ---")
print(response)

PyTorch version: 2.10.0+cu128
CUDA available for this GPU: True

Loading model on CPU (this takes 1-2 minutes)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model loaded successfully

Generating response...

--- Response ---
<|system|>You are a helpful assistant.<|user|> What is Linux Mint?<|assistant|> Linux Mint is a Linux operating system that is known for its user-friendly interface and its features for users with basic computer skills. It is a lightweight and easy-to-use operating system that offers a range of features and services that can meet the needs of both novice and experienced users. The operating system is based on the Lightweight Enterprise Linux kernel and is designed to be customizable and intuitive. Linux Mint is available in several editions, each with its own set of features, color schemes, and themes. Some popular editions of Linux Mint include:

1. Cinnamon: This edition has a light and minimalist interface, making it perfect for users looking for a more streamlined experience.

2. MATE: This edition has a dark, sleek, and minimalist interface, perfect for users who prefer a more traditional desktop environment.

3. 

In [ ]:
from transformers import TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from datasets import Dataset
import torch, json

# Load your data
with open("data.json") as f:
    raw = json.load(f)
dataset = Dataset.from_list(raw)

# LoRA config — keep rank (r) low to save memory
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def tokenize(example):
    text = example["instruction"] +" " + example["output"]
    return tokenizer(text, truncation=True, max_length=256, padding="max_length")

tokenized = dataset.map(tokenize)
tokenized = tokenized.map(lambda x: {"labels": x["input_ids"]})


args = TrainingArguments(
    output_dir="./tinyllama-finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=False,
    logging_steps=10,
    save_strategy="epoch",
    use_cpu=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
)
trainer.train()
    



/home/german1/ai-learning/notebooks/llm_env/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GeForce GTX 850M which is of cuda capability 5.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/home/german1/ai-learning/notebooks/llm_env/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/home/german1/ai-learning/notebooks/llm_env/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
NVIDIA GeForce GTX 850M with CUDA capability sm_50 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the NVIDIA GeForce GTX 850M GPU with PyTorch, please check the instru

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Step,Training Loss


In [ ]:
model.save_pretrained("./my-tinyllama-lora")
tokenizer.save_pretrained("./my-tinyllama-lora")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("./my-tinyllama-lora")

base_model=AutoModelForCausalLM.from_pretrained( "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    dtype=torch.float16,
    low_cpu_mem_usage=True
)

model = PeftModel.from_pretrained(base_model,"./my-tinyllama-lora")
model.eval()
print("model loaded")


while True:
    user_input = input("You: ")

    if user_input.strip().lower() in ["quit","exit","q"]:
        print("Goodbye!")
        break

        
    prompt = f"<|user|>{user_input}</s><|assistant|>"
    inputs = tokenizer(prompt, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )
        
        new_tokens = outputs[0][inputs["inputs_ids"].shape[1]:]
        response= tokenizer.decode(outputs[0],skip_special_tokens=True)
        print(f"Bot: {response}\n")

        del outputs, inputs, new_tokens
        torch.cuda.empty_cache()